# 🔑 Solutions: Lesson 3: Data Transformation
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
df = load_lesson_clean()

---
### Problem 3.1 &nbsp; ⭐ &nbsp; Scale by hand, verify with scikit-learn

Take the six rainfall values `[420, 560, 640, 700, 810, 900]`.

1. Compute the **min-max** scaled values and the **z-scores** using only NumPy.
2. Verify both against `MinMaxScaler` and `StandardScaler`.
3. Careful: NumPy's `std()` uses `ddof=0` by default. Which one does `StandardScaler` use?

In [2]:
x = np.array([420, 560, 640, 700, 810, 900], dtype=float)

mm = (x - x.min()) / (x.max() - x.min())
zs = (x - x.mean()) / x.std()                 # ddof=0: population std

mm_sk = MinMaxScaler().fit_transform(x.reshape(-1, 1)).ravel()
zs_sk = StandardScaler().fit_transform(x.reshape(-1, 1)).ravel()

print("min-max  :", mm.round(3), "| matches sklearn:", np.allclose(mm, mm_sk))
print("z-score  :", zs.round(3), "| matches sklearn:", np.allclose(zs, zs_sk))
print("StandardScaler divides by the POPULATION std (ddof=0) - same as np.std default.")

min-max  : [0.    0.292 0.458 0.583 0.812 1.   ] | matches sklearn: True
z-score  : [-1.598 -0.709 -0.201  0.18   0.878  1.45 ] | matches sklearn: True
StandardScaler divides by the POPULATION std (ddof=0) - same as np.std default.


**💬 Why this works:** The scalers are just two lines of arithmetic. Knowing that demystifies them. (pandas' `.std()` uses `ddof=1`, so if you standardise with pandas your numbers differ slightly from scikit-learn's: a classic source of confusion.)

---
### Problem 3.2 &nbsp; ⭐⭐ &nbsp; How badly does one giant farm squash min-max scaling?

Using `farm_size_ha` (drop NaN):

1. Min-max scale it. What fraction of farms end up with a scaled value **below 0.05**?
2. Do the same with `RobustScaler`. What are the 5th and 95th percentiles of the robust-scaled values?
3. Explain in one sentence why min-max is dangerous when outliers exist.

In [3]:
s = df[["farm_size_ha"]].dropna()
mm = MinMaxScaler().fit_transform(s).ravel()
rb = RobustScaler().fit_transform(s).ravel()

print(f"min-max: {np.mean(mm < 0.05) * 100:.1f}% of farms squeezed into [0, 0.05]  (max farm = {s.farm_size_ha.max():.0f} ha)")
print("robust 5th/95th percentile:", np.percentile(rb, [5, 95]).round(2))

min-max: 99.4% of farms squeezed into [0, 0.05]  (max farm = 378 ha)
robust 5th/95th percentile: [-0.61  2.86]


**💬 Why this works:** Min-max uses the *extremes* to define the scale. One 377-ha farm makes the whole 'normal' population look nearly identical (almost all values ≈ 0). Robust scaling uses the median and IQR, which the giants barely move.

---
### Problem 3.3 &nbsp; ⭐⭐ &nbsp; Skew hunting

1. Compute the skewness of every numeric column in the lesson-clean data and list the **three most right-skewed** columns.
2. Apply `np.log1p` to them and report the new skewness.
3. For which of the three did the log help the *least*, and why might that be?

In [4]:
skew = df.select_dtypes("number").skew().sort_values(ascending=False)
top3 = skew.head(3).index.tolist()
print("most skewed:\n", skew.head(3).round(2).to_string())

after = np.log1p(df[top3]).skew()
print("\nafter log1p:\n", after.round(2).to_string())

most skewed:
 farm_size_ha         14.36
nitrogen_ppm          7.68
total_revenue_ksh     4.76

after log1p:
 farm_size_ha         3.26
nitrogen_ppm        -0.61
total_revenue_ksh   -0.70


**💬 Why this works:** Log tamed revenue and nitrogen, but `farm_size_ha` stays skewed (≈ 3): a few 300-ha giants still stretch the tail. Log works best on *multiplicative* skew; when a handful of values are extreme (genuine giants or errors) you may also need capping or a fitted power transform such as Yeo-Johnson.

---
### Problem 3.4 &nbsp; ⭐⭐ &nbsp; The ordinal-encoding trap

`pest_pressure` is ordinal: Low < Medium < High.

1. Fit scikit-learn's default `OrdinalEncoder` on it. What numbers do Low, Medium and High get? Is that order sensible?
2. Fix it by specifying the order explicitly.
3. As evidence, compute the mean yield for each pest level and confirm the ordering matches the pressure.

In [5]:
pest = df[["pest_pressure"]].dropna()

default = OrdinalEncoder().fit(pest)
print("default order :", dict(zip(default.categories_[0], range(3))), "  <- High=0, Low=1, Medium=2 !!")

fixed = OrdinalEncoder(categories=[["Low", "Medium", "High"]]).fit(pest)
print("explicit order:", dict(zip(fixed.categories_[0], range(3))))

print("\nmean yield by pest level:")
print(df.groupby("pest_pressure")["yield_t_ha"].mean().reindex(["Low", "Medium", "High"]).round(2))

default order : {'High': 0, 'Low': 1, 'Medium': 2}   <- High=0, Low=1, Medium=2 !!
explicit order: {'Low': 0, 'Medium': 1, 'High': 2}

mean yield by pest level:
pest_pressure
Low       2.83
Medium    2.38
High      1.83
Name: yield_t_ha, dtype: float64


**💬 Why this works:** Alphabetical order puts 'Medium' above 'High'. A model would learn nonsense. Always pass `categories=` for ordinal features. The yield means (Low > Medium > High) confirm the real-world order we encoded.

---
### Problem 3.5 &nbsp; ⭐⭐⭐ &nbsp; One-hot encoding in the real world: unseen categories

1. Split the data 80/20 (`random_state=0`). Fit a `OneHotEncoder` on the **training** `county` column only.
2. Create a small 'new data' frame with 3 rows where the first row's county is `'Narok'` (never seen in training). Transform it with (a) `pd.get_dummies` and (b) your fitted encoder. What happens with each?
3. How many columns does the encoder produce, and how would `drop='first'` change that?

In [6]:
train, test = train_test_split(df, test_size=0.2, random_state=0)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[["county"]])

new = test.head(3).copy()
new.iloc[0, new.columns.get_loc("county")] = "Narok"            # a county the model has never seen

print("get_dummies columns on new data:", list(pd.get_dummies(new["county"]).columns))
print("encoder output shape           :", ohe.transform(new[["county"]]).shape)
print("row 0 (unseen county)          :", ohe.transform(new[["county"]])[0])
print("columns:", len(ohe.get_feature_names_out()),
      "| with drop='first':", OneHotEncoder(drop="first").fit(train[["county"]]).get_feature_names_out().shape[0])

get_dummies columns on new data: ['Bungoma', 'Machakos', 'Narok']
encoder output shape           : (3, 10)
row 0 (unseen county)          : [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
columns: 10 | with drop='first': 9


**💬 Why this works:** `pd.get_dummies` builds columns from whatever it sees, so new data produces a *different set of columns* than training → the model breaks. A fitted `OneHotEncoder` remembers the training categories and (with `handle_unknown='ignore'`) encodes an unseen category as all zeros.